<a href="https://colab.research.google.com/github/zbovaird/UHG-Library/blob/uhg-dev/colab/uhg_cic_intrusion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UHG Intrusion Detection on CIC Network Flow Data

## Purpose

Intrusion detection on CIC network flow data using **Universal Hyperbolic Geometry (UHG)**. UHG embeds data in hyperbolic space where hierarchical structure (e.g., attack vs. benign) is naturally captured.

## Methodology

- **Part A (Unsupervised)**: Build kNN graph (FAISS CPU or PyNNDescent fallback, optional PCA **only for neighbor search**), train ProjectiveGraphSAGE for UHG embeddings, cluster with DBSCAN, score anomalies by centroid/neighbor quadrance. No labels used for training; ideal for discovering unknown attacks.

- **Part B (Supervised)**: Same graph + ProjectiveGraphSAGE, but train with CrossEntropyLoss for multi-class classification. Use when labeled data is available and you need discrete attack-type predictions.

- **Memory (aligned with `tests/UHG_IDS_4_9.ipynb`)**: Version-matched **torch-scatter** wheel, **faiss-cpu** + **pynndescent** for kNN, and **PCA** to reduce dimensionality for the index only—full tabular features still feed the GNN.

## Data

CIC dataset at `/content/drive/MyDrive/CIC_data.csv`. Subsample fraction is configurable (`SAMPLE_FRAC` / `UHG_SAMPLE_FRAC`); large runs need enough RAM for the sampled table plus graph build.

## Outputs

- **(A)** Anomaly scores, top-k rankings, exported detector
- **(B)** Classifier accuracy, saved model weights

In [ ]:
# Colab: PyG deps, version-matched torch-scatter, FAISS CPU, PyNNDescent, then UHG (uhg-dev).
# Local Jupyter: install the same extras once (see docs/development/colab-local.md) or pip install -e ".[colab]"

import os
import sys
import subprocess

UHG_GIT_REF = os.environ.get("UHG_GIT_REF", "uhg-dev")


def _colab_install_extras():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "torch_geometric", "scikit-learn", "scipy", "pandas", "pyarrow", "tqdm",
    ])
    import torch

    print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
    TORCH = torch.__version__.split("+")[0]
    if torch.cuda.is_available() and torch.version.cuda:
        CUDA = "cu" + torch.version.cuda.replace(".", "")
        idx = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"
    else:
        idx = f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html"
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "torch-scatter", "-f", idx,
    ])
    try:
        import faiss
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pynndescent"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        f"git+https://github.com/zbovaird/UHG-Library.git@{UHG_GIT_REF}",
    ])


if "google.colab" in sys.modules:
    _colab_install_extras()

import torch
import uhg

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

FILE_PATH = os.environ.get("CIC_CSV", "/content/drive/MyDrive/CIC_data.csv")
MODEL_SAVE_PATH = os.environ.get("UHG_MODEL_OUT", "/content/drive/MyDrive/uhg_ids_model.pth")
DETECTOR_SAVE_PATH = os.environ.get("UHG_DETECTOR_OUT", "/content/drive/MyDrive/uhg_ids_detector.pt")
RESULTS_PATH = os.environ.get("UHG_RESULTS_DIR", "/content/drive/MyDrive/uhg_ids_results")
os.makedirs(RESULTS_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU — training may be slower")

assert hasattr(uhg, "__version__"), "uhg must expose __version__"
print("Installed UHG ref:", UHG_GIT_REF)
print("UHG version:", uhg.__version__)
HAS_ANOMALY = getattr(uhg, "UHGUnsupervisedAnomalyDetector", None) is not None
print("Part A (unsupervised) available:", HAS_ANOMALY)
assert HAS_ANOMALY, "Install a UHG build that includes UHGUnsupervisedAnomalyDetector"


In [ ]:
# Load CIC data: fast CSV via PyArrow; stratified floor sampling.
# Graph build (Part A/B): PCA is applied only inside build_knn_graph / fit() for neighbor search.

import os
import pandas as pd
import numpy as np

from uhg.utils.cic_io import read_cic_csv
from uhg.utils.sampling import stratified_subsample_indices

SAMPLE_FRAC = float(os.environ.get("UHG_SAMPLE_FRAC", "0.10"))
PCA_COMPONENTS = int(os.environ.get("UHG_PCA_COMPONENTS", "20"))
KNN_BACKEND = os.environ.get("UHG_KNN_BACKEND", "auto")

print(f"Config: SAMPLE_FRAC={SAMPLE_FRAC}, PCA_COMPONENTS={PCA_COMPONENTS} (KNN only), KNN_BACKEND={KNN_BACKEND}")

print(f"Loading data from: {FILE_PATH}")
data = read_cic_csv(FILE_PATH)
data.columns = data.columns.str.strip()
if "Label" in data.columns:
    data["Label"] = data["Label"].str.strip()

unique_labels = data["Label"].unique() if "Label" in data.columns else []
print(f"\nUnique labels: {len(unique_labels)}")
if "Label" in data.columns:
    print(data["Label"].value_counts())

n_target = max(1, int(SAMPLE_FRAC * len(data)))
min_floor = min(10, n_target // max(1, len(unique_labels)))
try:
    idx = stratified_subsample_indices(
        data["Label"].values, n_total=n_target, min_per_class=max(1, min_floor), random_state=42
    )
    data_sampled = data.iloc[idx].reset_index(drop=True)
except Exception as e:
    print("Stratified sample fallback (random):", e)
    data_sampled = data.sample(frac=SAMPLE_FRAC, random_state=42)

try:
    from uhg.utils.schema import detect_label_column, enforce_numeric
    label_col = detect_label_column(data_sampled)
    feature_cols = [c for c in data_sampled.columns if c != label_col]
    df_features = data_sampled[feature_cols].copy()
    df_features = enforce_numeric(df_features, fill="mean", replace_inf=True)
    labels_series = data_sampled[label_col] if label_col else None
except ImportError:
    df_features = data_sampled.drop(columns=["Label"], errors="ignore").apply(
        pd.to_numeric, errors="coerce"
    )
    df_features = df_features.replace([np.inf, -np.inf], np.nan)
    df_features = df_features.fillna(df_features.mean()).fillna(0)
    labels_series = data_sampled["Label"] if "Label" in data_sampled.columns else None

print(f"\nSampled shape: {df_features.shape}")
df_sampled = df_features.copy()
if labels_series is not None:
    df_sampled["Label"] = labels_series.values



In [ ]:
# Part A: Unsupervised anomaly detection (FAISS/PyNNDescent kNN + optional PCA for neighbors).

Detector = uhg.UHGUnsupervisedAnomalyDetector
detector = Detector(hidden=64, embedding_dim=32)
detector.fit_from_dataframe(
    df_sampled,
    epochs=50,
    k=5,
    seed=42,
    undirected=True,
    pca_components=PCA_COMPONENTS,
    knn_backend=KNN_BACKEND,
)

try:
    from uhg.cluster.dbscan import auto_eps_kdist
    eps_suggested = auto_eps_kdist(detector.embeddings.numpy(), k=4)
    print(f"Suggested eps (k-dist): {eps_suggested:.4f}")
except Exception:
    pass

detector.cluster(eps=0.5, min_samples=3)
scores = detector.score(method="centroid_quadrance")
summary = detector.summarize(topk=20)
print("Timings:", summary.get("timings", {}))
print("Top entities:", summary.get("top_entities", [])[:10])
if summary.get("cluster_metrics"):
    print("Cluster metrics:", summary["cluster_metrics"])

detector.export(DETECTOR_SAVE_PATH)
print(f"Detector exported to {DETECTOR_SAVE_PATH}")


In [ ]:
# Part A evaluation: compare top-k anomalies to ground-truth labels (when available).
# Not used for training - only for post-hoc evaluation.

if HAS_ANOMALY and 'Label' in df_sampled.columns:
    try:
        scores = detector.score(method='centroid_quadrance')
        top_indices = torch.topk(scores, min(100, len(scores))).indices.numpy()
        labels_arr = df_sampled['Label'].values
        benign_key = 'BENIGN'
        attack_count = sum(1 for i in top_indices if labels_arr[i] != benign_key)
        print(f"Of top-{len(top_indices)} anomalies, {attack_count} were attacks (non-BENIGN)")
    except NameError:
        print("Run Part A cell first to define detector")

In [ ]:
# Part B: supervised classification (optional focal loss + undirected kNN).

if "Label" not in df_sampled.columns:
    print("No Label column - skipping Part B")
else:
    try:
        from sklearn.preprocessing import StandardScaler
        from uhg.graph.build import build_knn_graph
        from uhg.nn.models.sage import ProjectiveGraphSAGE
        from uhg.nn.early_stopping import EarlyStopping
        from uhg.nn.losses import FocalLoss

        scaler = StandardScaler()
        X = scaler.fit_transform(df_features.values.astype(np.float32))
        label_mapping = {lb: i for i, lb in enumerate(unique_labels)}
        y = np.array([label_mapping.get(lb, 0) for lb in labels_series])

        USE_UNDIRECTED_KNN = True
        USE_FOCAL = False  # set True if classes are heavily imbalanced

        edge_index = build_knn_graph(
            X,
            k=5,
            undirected=USE_UNDIRECTED_KNN,
            pca_components=PCA_COMPONENTS,
            knn_backend=KNN_BACKEND,
        )
        x_t = torch.tensor(X, dtype=torch.float32).to(device)
        edge_index = edge_index.to(device)
        y_t = torch.tensor(y, dtype=torch.long).to(device)

        n = len(x_t)
        perm = torch.randperm(n, device=device)
        train_end = int(0.7 * n)
        val_end = train_end + int(0.15 * n)
        train_mask = torch.zeros(n, dtype=torch.bool, device=device)
        val_mask = torch.zeros(n, dtype=torch.bool, device=device)
        test_mask = torch.zeros(n, dtype=torch.bool, device=device)
        train_mask[perm[:train_end]] = True
        val_mask[perm[train_end:val_end]] = True
        test_mask[perm[val_end:]] = True

        in_ch = x_t.size(1)
        num_classes = len(label_mapping)
        model = ProjectiveGraphSAGE(
            in_channels=in_ch,
            hidden_channels=128,
            out_channels=num_classes,
            num_layers=2,
            dropout=0.2,
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
        criterion = FocalLoss(gamma=2.0) if USE_FOCAL else torch.nn.CrossEntropyLoss()
        early_stop = EarlyStopping(min_delta=0.01, patience=20, mode="max")

        for epoch in range(1, 301):
            model.train()
            optimizer.zero_grad()
            out = model(x_t, edge_index)
            loss = criterion(out[train_mask], y_t[train_mask])
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                pred = model(x_t, edge_index).argmax(dim=1)
                val_acc = (pred[val_mask] == y_t[val_mask]).float().mean().item()
            if early_stop(val_acc, model):
                print(f"Early stopping at epoch {epoch}")
                break
            if epoch % 20 == 0:
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Val Acc: {val_acc:.4f}")

        early_stop.restore_best(model)
        model.to(device)
        test_acc = (model(x_t, edge_index).argmax(dim=1)[test_mask] == y_t[test_mask]).float().mean().item()
        print(f"Final test accuracy: {test_acc:.4f}")
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"Model saved to {MODEL_SAVE_PATH}")
    except ImportError as e:
        print(f"Part B import error: {e}")



## Summary

- **Part A (Unsupervised)**: Use when you want to discover anomalies without labels. Outputs anomaly scores, top-k rankings, and an exported detector. Requires UHG 0.3.7+.

- **Part B (Supervised)**: Use when you have labeled data and need attack-type classification. Outputs test accuracy and saved model weights. Requires UHG 0.3.7+ for library components.

Both workflows use the same CIC data path and save outputs to Google Drive.